# Inspect Firestore collections

Print a few sample documents and check the size of a Firestore collection, working from
its local export (gzipped JSON lines, one document per line) rather than querying Firestore.

In [ ]:
import gzip
import json
from collections import Counter
from pathlib import Path

EXPORT_PATH = Path("../profile_compensation_estimates.jsonl.gz")
N_SAMPLES = 3  # documents to print

with gzip.open(EXPORT_PATH, "rt", encoding="utf-8") as f:
    lines = [line for line in f if line.strip()]
records = [json.loads(line) for line in lines]

print(f"loaded {len(records):,} records from {EXPORT_PATH.name}")

## Sample documents

In [ ]:
def _truncate(value, max_len=200, max_items=2):
    """Shorten long strings and lists so a printed document stays readable."""
    if isinstance(value, str) and len(value) > max_len:
        return value[:max_len] + f"... (+{len(value) - max_len} chars)"
    if isinstance(value, list):
        head = [_truncate(v, max_len, max_items) for v in value[:max_items]]
        return head + [f"... (+{len(value) - max_items} more)"] if len(value) > max_items else head
    if isinstance(value, dict):
        return {k: _truncate(v, max_len, max_items) for k, v in value.items()}
    return value


for record in records[:N_SAMPLES]:
    print(f"--- {record.get('doc_id')} ---")
    print(json.dumps(_truncate(record), indent=2, ensure_ascii=False))

## Collection size

In [ ]:
doc_ids = [r.get("doc_id") for r in records]
# A resumed export can write the same page twice, so check for repeats.
print(f"records:         {len(records):,}")
print(f"unique doc_ids:  {len(set(doc_ids)):,}")

record_bytes = [len(line.encode("utf-8")) for line in lines]
total_bytes = sum(record_bytes)
print(f"compressed file: {EXPORT_PATH.stat().st_size / 1e6:,.1f} MB")
print(f"uncompressed:    {total_bytes / 1e6:,.1f} MB")
print(f"per record:      avg {total_bytes / len(lines):,.0f} B, min {min(record_bytes):,} B, max {max(record_bytes):,} B")

## Fields

Which fields appear, how often, and with what types.

In [ ]:
field_types = {}
for record in records:
    for key, value in record.items():
        field_types.setdefault(key, Counter())[type(value).__name__] += 1

for key, types in field_types.items():
    print(f"{key:<15} {sum(types.values()):>7,}/{len(records):,}  {dict(types)}")